# TTA experiment: prediction evolution over training, with/without color TTA views

Trains the full-data model (all 24 labeled images, no held-out split -- same setup as
`scripts/train.py`) for up to **100 epochs**, and after *every* epoch renders the current
prediction on all 6 real test images under three inference modes:

1. **Plain** -- a single forward pass.
2. **TTA (geometric)** -- 8-way dihedral averaging (horizontal flip x 90-degree rotation).
   Exact, invertible pixel permutations -- see `roof_seg/tta.py` module docstring.
3. **TTA (geometric + color)** -- adds `roof_seg.tta.DEFAULT_COLOR_VARIANTS`
   (brightness/gamma perturbations). These need no inverse mapping since they never move
   a pixel, only its value -- a color-perturbed view's prediction is already pixel-aligned.

**Why not perspective/resized_crop as TTA views:** both require resampling/cropping, so an
exact per-pixel inverse for their *predictions* isn't recoverable without reintroducing
interpolation -- the same reason those two augmentation features need `cv2.INTER_NEAREST`
special-casing to stay mask-safe during *training* at all. TTA only uses transform families
with an exact inverse (geometric) or no geometric effect at all (color).

There is no validation split here (matches the final deliverable's training regime), so
there is no automatic best-checkpoint/early-stopping signal -- "overfitting" has to be read
qualitatively from how the per-epoch prediction grids evolve (e.g. does TTA's extra
smoothing start disagreeing with the plain prediction in ways that look like noise rather
than refinement?), not from a metric. For a *quantitative* best-epoch estimate see the
6-fold CV ensemble comparison already run in `scripts/compare_augmentation_ensemble.py`
(SPEC.md 3.5), which found best epochs averaging ~18-21 out of a 25-epoch budget.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from roof_seg.config import IMAGENET_MEAN, IMAGENET_STD, INSPECTION_DIR, RANDOM_SEED  # noqa: E402
from roof_seg.dataset import RoofTestDataset, get_train_ids  # noqa: E402
from roof_seg.seed import set_seed  # noqa: E402
from roof_seg.train import TrainConfig, get_device, train_model  # noqa: E402
from roof_seg.tta import DEFAULT_COLOR_VARIANTS, predict_with_tta  # noqa: E402

## Config

`EPOCHS` is the upper budget requested (up to 100). `PREVIEW_EVERY` controls how often a
grid is rendered/saved -- 1 means every epoch, as requested; raise it if 100 epochs x
6 images x (1 + 8 + 12) forward passes turns out too slow on your machine (CPU: expect this
to be considerably slower than plain training -- each preview alone is ~21 extra forward
passes per image, 126 per epoch across all 6 test images).

In [ ]:
EPOCHS = 100
PREVIEW_EVERY = 1  # render a grid after every epoch, as requested
SEED = RANDOM_SEED
OUT_DIR = INSPECTION_DIR / "tta_experiment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = get_device()  # roof_seg.train.get_device(): cuda if torch.cuda.is_available() else cpu

print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("""\nNo GPU visible to PyTorch -- running on CPU. If you expected a GPU here, check:
  1. `nvidia-smi` works inside this container (if not, it wasn't started with GPU access,
     e.g. missing `--gpus all` / `--runtime=nvidia` on `docker run`).
  2. The installed torch build has CUDA support: `python -c "import torch; print(torch.version.cuda)"`
     -- `None` means a CPU-only wheel was installed; reinstall from
     https://pytorch.org/get-started/locally/ with a CUDA version matching the container's driver.\n""")
print(f"Device: {device}")


## Per-epoch preview callback

`train_model()` (from `roof_seg/train.py`) accepts an `epoch_callback(epoch, model)` fired
after every completed epoch, with the model in `eval()` mode -- purely a side effect, it
never influences training or checkpointing (see SPEC.md 3.7). Here it renders a 3-row
(plain / TTA-geometric / TTA-geometric+color) x 6-column (test images) overlay grid, shows
it inline, and saves it to `outputs/inspection/tta_experiment/epoch_NNN.png`.

In [ ]:
test_ds = RoofTestDataset()
TEST_IDS_ORDERED = [test_ds[i]["id"] for i in range(len(test_ds))]


def denormalize(image_tensor: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    unnormalized = (image_tensor * std + mean).clamp(0, 1)
    return (unnormalized.permute(1, 2, 0).numpy() * 255).astype(np.uint8)


def overlay(image: np.ndarray, mask: np.ndarray, color=(255, 0, 0)) -> np.ndarray:
    out = image.copy()
    out[mask] = (0.5 * out[mask] + 0.5 * np.array(color)).astype(np.uint8)
    return out


@torch.no_grad()
def render_epoch_preview(epoch: int, model: torch.nn.Module, save: bool = True, show: bool = True):
    n = len(test_ds)
    row_labels = ["plain", "TTA (geometric)", "TTA (geometric+color)"]
    fig, axes = plt.subplots(len(row_labels), n, figsize=(2.4 * n, 3.6 * len(row_labels)))

    for col in range(n):
        sample = test_ds[col]
        image_tensor = sample["image"]
        image_np = denormalize(image_tensor)

        plain_probs = torch.sigmoid(model(image_tensor.unsqueeze(0).to(device)))[0, 0].cpu().numpy()
        plain_mask = plain_probs > 0.5

        tta_geo_probs = predict_with_tta(model, image_tensor, device=device)[0].numpy()
        tta_geo_mask = tta_geo_probs > 0.5

        tta_color_probs = predict_with_tta(
            model, image_tensor, device=device, color_variants=DEFAULT_COLOR_VARIANTS
        )[0].numpy()
        tta_color_mask = tta_color_probs > 0.5

        axes[0, col].imshow(overlay(image_np, plain_mask))
        axes[1, col].imshow(overlay(image_np, tta_geo_mask))
        axes[2, col].imshow(overlay(image_np, tta_color_mask))

        for row in range(len(row_labels)):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])
        axes[0, col].set_title(sample["id"], fontsize=9)

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=9)

    fig.suptitle(f"Epoch {epoch}/{EPOCHS}", fontsize=12)
    fig.tight_layout()

    if save:
        fig.savefig(OUT_DIR / f"epoch_{epoch:03d}.png", dpi=100)
    if show:
        clear_output(wait=True)
        display(fig)
    plt.close(fig)


def epoch_callback(epoch: int, model: torch.nn.Module) -> None:
    if epoch == 1 or epoch % PREVIEW_EVERY == 0 or epoch == EPOCHS:
        render_epoch_preview(epoch, model)

## Train

All 24 labeled images, no held-out split (matches the final deliverable's training regime --
SPEC.md 3.7), `early_stopping_patience` has no effect without a validation split so the full
`EPOCHS` budget always runs. Augmentation stays on with the confirmed `DEFAULT_FEATURES`
(flip, rotate90, color_jitter) -- see SPEC.md 3.5's ensemble re-confirmation.

In [ ]:
config = TrainConfig(epochs=EPOCHS, seed=SEED)  # augment=True, augment_features=None -> DEFAULT_FEATURES

result = train_model(
    train_ids=get_train_ids(),
    val_ids=None,
    config=config,
    progress=True,
    device=device,
    epoch_callback=epoch_callback,
)

## Training loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, len(result.train_losses) + 1), result.train_losses)
ax.set_xlabel("epoch")
ax.set_ylabel("train loss (BCE + Dice)")
ax.set_title("Training loss")
fig.tight_layout()
plt.show()

## Notes for interpreting the per-epoch grids

- Early epochs (see the epoch-1 grid saved to `outputs/inspection/tta_experiment/epoch_001.png`)
  typically predict most of the frame as roof, before the model has learned to discriminate
  (compare `scripts/train_with_test_preview.py`'s documented run, which shows the same pattern
  through epoch 6).
- Watch where the three rows start to **diverge**: TTA views averaging away real detail (not
  just boundary noise) is a sign the single-view predictions have become overconfident in a
  way TTA is smoothing over -- worth a closer look at those epochs specifically.
- With no validation split, there is no ground truth to score these predictions against --
  this run is for *qualitative* inspection only. Cross-reference against the CV ensemble's
  quantitative best-epoch estimates (SPEC.md 3.5: ~18-21 epochs on a 25-epoch budget with a
  real validation split) before deciding this run's checkpoint at some specific epoch is
  actually better than the current deliverable.